In [1]:
# ============================================================
# 3D Fusion (Tokamak) — MHD-Inspired v2 + URT Controller (Colab)
# ============================================================
# What this version adds (vs. the toy demo):
# - PDE-like plasma evolution with advection (shear flow), diffusion, and nonlinear saturation
# - q-profile and magnetic shear shaping the toroidal/poloidal advection
# - Resistive Wall Mode (RWM)–like growth with wall time constant τ_wall
# - Mode coupling (nonlinear), noise, and disruption events
# - 4 external coils; URT controller drives coil currents to suppress edge modes
# - Auto-stability safeguard for URT (keeps κ<1) + optional inner loops
# - Plotly isosurface (interactive) with fallback to Matplotlib; metrics/diagnostics
# ------------------------------------------------------------
# NOTE: This is a didactic research sandbox — not a physical MHD solver.

import numpy as np
import time, math, sys, warnings

# ----------------- Optional Plotly for 3D isosurface -----------------
try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except Exception:
    HAS_PLOTLY = False

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa

# ------------------------ Utility helpers ----------------------------
def safe_norm(x, eps=1e-12):
    return np.sqrt(np.sum(x*x) + eps)

def roll3(A, s_i=0, s_j=0, s_k=0):
    if s_i: A = np.roll(A, s_i, axis=0)
    if s_j: A = np.roll(A, s_j, axis=1)
    if s_k: A = np.roll(A, s_k, axis=2)
    return A

# ------------------------ URT Controller -----------------------------
class FastURT:
    """
    Minimal, stable URT update for a control vector u ∈ R^m:
      u_{k+1} = β * [ α( u_k - θ_h * φ(u_k) ) + v_k ],
    where v_k is a 'learning signal' (here: mapped from measured mode amplitudes).
    - φ() is smooth odd; we use tanh for saturation & good gradients.
    - Auto-stability guard forces κ = β α (1+θ_h) < κ_max < 1.
    - Optional inner_loops for extra contraction per outer step.
    """
    def __init__(self, alpha=1.0, theta_h=2.0, beta=0.30, kappa_max=0.95, inner_loops=1):
        self.alpha   = float(alpha)
        self.theta_h = float(theta_h)
        self.beta    = float(beta)
        self.inner_loops = int(max(1, inner_loops))
        # Auto-stability clamp:
        kappa = self.beta*self.alpha*(1.0+self.theta_h)
        if kappa >= kappa_max:
            self.beta = (kappa_max - 1e-6) / (self.alpha*(1.0+self.theta_h))
            kappa = self.beta*self.alpha*(1.0+self.theta_h)
        print(f"Stability verified: κ={kappa:.3f}  (β={self.beta:.3f}, inner_loops={self.inner_loops})")

    def phi(self, x):
        return np.tanh(x)

    def _one_step(self, u, v):
        return self.beta * ( self.alpha*(u - self.theta_h*self.phi(u)) + v )

    def step(self, u, v):
        # Optional inner fixed-point sweeps for stronger contraction
        for _ in range(self.inner_loops):
            u = self._one_step(u, v)
        return u

# ----------------------- MHD-Inspired Tokamak ------------------------
class Tokamak3D_MHD:
    """
    3D torus on structured grid (r, θ, φ) with scalar pressure-like field P.
    Evolution (discrete time):
      P <- P + Δt * [
          - v⋅∇P            (advection from shear flow tied to q-profile)
          + D ∇²P           (diffusion)
          + S_modes         (low-m/n harmonics + RWM drive)
          - μ P^3           (nonlinear saturation)
        ] + noise + coil_field

    Features:
    - q(r) profile; advection velocities v_θ(r), v_φ(r) ∝ 1/q(r)
    - Resistive Wall Mode-like growth on edge with τ_wall
    - Nonlinear coupling between selected modes
    - 4 coils mapped to harmonics; controller sets coil currents
    """

    def __init__(self, R0=1.7, a=0.5, Nr=24, Nth=48, Nph=48, seed=0,
                 D=5e-3, mu=0.08, dt=1.0,
                 q0=1.2, q_edge=3.8, rwm_gain=0.03, tau_wall=150.0,
                 noise_sigma=2e-4, couple_strength=0.05):
        rng = np.random.default_rng(seed)
        self.R0, self.a = float(R0), float(a)
        self.Nr, self.Nth, self.Nph = int(Nr), int(Nth), int(Nph)
        self.D, self.mu, self.dt = float(D), float(mu), float(dt)
        self.q0, self.q_edge = float(q0), float(q_edge)
        self.rwm_gain, self.tau_wall = float(rwm_gain), float(tau_wall)
        self.noise_sigma = float(noise_sigma)
        self.couple_strength = float(couple_strength)

        # Grids
        r  = np.linspace(0, a, Nr)
        th = np.linspace(0, 2*np.pi, Nth, endpoint=False)
        ph = np.linspace(0, 2*np.pi, Nph, endpoint=False)
        self.r, self.th, self.ph = r, th, ph
        R, TH, PH = np.meshgrid(r, th, ph, indexing='ij')
        self.R, self.TH, self.PH = R, TH, PH

        # Cartesian for rendering
        X = (R0 + R*np.cos(TH)) * np.cos(PH)
        Y = (R0 + R*np.cos(TH)) * np.sin(PH)
        Z = R * np.sin(TH)
        self.X, self.Y, self.Z = X, Y, Z

        # Base equilibrium
        base = np.exp(-(R**2) / (0.85*a)**2)
        self.P = base.copy()

        # q-profile (monotonic example)
        #   q(r) = q0 + (q_edge - q0)*(r/a)^2
        self.q_profile = self.q0 + (self.q_edge - self.q0)*(self.R/self.a)**2

        # Advection velocities ~ poloidal & toroidal flows (unitless)
        # Increase edge shear via 1/q(r) and radial weighting
        v_phi_mag = 0.15 / (self.q_profile + 1e-6)         # toroidal angular speed
        v_th_mag  = 0.10 * (1.0 + 0.5*(self.R/self.a))     # poloidal angular speed
        # Convert to index-space "shift per step" (CFL-ish discrete advection)
        self.vk_phi = v_phi_mag * self.Nph * self.dt / (2*np.pi)  # cells per step
        self.vk_th  = v_th_mag  * self.Nth * self.dt / (2*np.pi)  # cells per step

        # RWM driver state (scalar proxy at edge)
        self.rwm_state = 0.0
        self.r_edge_mask = (self.R > 0.7*self.a).astype(float)

        # Low-order modes & initial phases
        self.modes = [(1,1), (2,1), (2,2)]
        self.phi0 = {k: rng.uniform(0, 2*np.pi) for k in self.modes}

        # 4 coils (φ = 0, π/2, π, 3π/2), θ = 0 midplane; distance just outside plasma
        self.num_coils = 4
        self.coil_phi = np.array([0, 0.5*np.pi, np.pi, 1.5*np.pi])
        self.coil_R   = R0 + a + 0.05
        self.u = np.zeros(self.num_coils)

        self.coil_fields = self._build_coil_fields()

        # Growth rates baseline (for the explicit "source" S_modes)
        self.gamma0 = {(1,1): 0.010, (2,1): 0.006, (2,2): 0.004}
        # History
        self.h = {"t":[], "A11":[], "A21":[], "A22":[], "u_norm":[],
                  "rwm":[]}

    # -------------------- Spatial helpers --------------------
    def _build_coil_fields(self):
        fields = []
        for phi_c in self.coil_phi:
            xc = (self.coil_R) * np.cos(phi_c)
            yc = (self.coil_R) * np.sin(phi_c)
            zc = 0.0
            dx, dy, dz = self.X - xc, self.Y - yc, self.Z - zc
            dist2 = dx*dx + dy*dy + dz*dz
            f = 1.0 / (dist2 + 1e-3)
            f /= np.max(f)
            fields.append(f)
        return np.array(fields)  # (4, Nr, Nth, Nph)

    def _harmonic_basis(self, m, n):
        # Edge-weighted cosine basis
        edge_w = np.clip(self.R/self.a, 0, 1)
        return edge_w * np.cos(m*self.TH + n*self.PH + self.phi0[(m,n)])

    def _laplacian(self, A):
        # Simple 3D periodic laplacian in θ/φ and reflecting at r edges
        # radial second difference with reflecting boundary at r=0,a
        Arp = roll3(A, s_i=+1); Arm = roll3(A, s_i=-1)
        Athp= roll3(A, s_j=+1); Athm= roll3(A, s_j=-1)
        Aphp= roll3(A, s_k=+1); Aphpm=roll3(A, s_k=-1)
        lap_r   = (Arp - 2*A + Arm)
        lap_th  = (Athp - 2*A + Athm)
        lap_ph  = (Aphp - 2*A + Aphpm)
        return lap_r + lap_th + lap_ph

    def _advect(self, A):
        # Upwind-like integer/cell shift advection using local velocities (vk_th, vk_phi)
        # Half-integer rounding to reduce bias
        shift_th  = np.rint(self.vk_th ).astype(int)
        shift_phi = np.rint(self.vk_phi).astype(int)
        out = A.copy()
        # θ advection (varying by r,φ through vk_th tensor)
        for i in range(self.Nr):
            for k in range(self.Nph):
                s = int(shift_th[i,0,k] if shift_th.ndim==3 else shift_th[i,k])
                if s != 0:
                    out[i,:,k] = np.roll(out[i,:,k], -s)
        # φ advection (varying by r,θ)
        for i in range(self.Nr):
            for j in range(self.Nth):
                s = int(shift_phi[i,j,0] if shift_phi.ndim==3 else shift_phi[i,j])
                if s != 0:
                    out[i,j,:] = np.roll(out[i,j,:], -s)
        return out

    # -------------------- Diagnostics --------------------
    def measure_modes(self):
        amps = {}
        for (m,n) in self.modes:
            B = self._harmonic_basis(m,n)
            w = self.r_edge_mask
            num = np.sum(self.P*B*w)
            den = np.sum(B*B*w) + 1e-12
            amps[(m,n)] = float(num/den)
        return amps

    def coil_to_harmonics_map(self):
        M = []
        for (m,n) in self.modes:
            B = self._harmonic_basis(m,n)
            row = [np.sum(self.coil_fields[i]*B*self.r_edge_mask)
                   for i in range(self.num_coils)]
            M.append(row)
        M = np.array(M)
        M /= (np.max(np.abs(M)) + 1e-9)
        return M  # (3,4)

    # ------------------ Physics source terms -----------------
    def _rwm_drive(self, amps):
        # Simple RWM proxy: driven by edge pressure & q_edge proximity
        edge_p = np.sum(self.P*self.r_edge_mask) / (np.sum(self.r_edge_mask) + 1e-9)
        # growth stronger when q_edge near integer (e.g., m/n ~ 2/1 → q~2)
        q_edge_now = float(np.mean(self.q_profile[self.r_edge_mask>0]))
        proximity  = min(1.0, 1.0/ (0.1 + abs(q_edge_now - 2.0)))  # peak ~ at q≈2
        # Discrete time update for rwm_state (leaky integrator with τ_wall)
        drwm = self.dt*( self.rwm_gain*edge_p*proximity - self.rwm_state/self.tau_wall )
        self.rwm_state += drwm
        # RWM acts mainly on (2,1) / (1,1)
        rw11 =  0.6*self.rwm_state
        rw21 =  1.0*self.rwm_state
        rw22 =  0.3*self.rwm_state
        return {(1,1):rw11, (2,1):rw21, (2,2):rw22}

    def _mode_coupling(self, amps):
        # Quadratic coupling (nonlinear) between modes
        # Example: A(2,1) feeds A(1,1) and A(2,2), etc.
        a11, a21, a22 = amps[(1,1)], amps[(2,1)], amps[(2,2)]
        c = self.couple_strength
        return {(1,1): c*(a21*a22),
                (2,1): c*(a11*a22),
                (2,2): c*(a11*a21)}

    # ------------------------ One step -----------------------
    def step(self, enable_growth=True, enable_control=True,
             coil_gain=0.08, noise_on=True, disruption=None):
        """
        disruption: optional dict like
          {'step': k, 'delta': +0.5, 'region':'edge'/'core'/'all'}
        """
        P = self.P

        # (1) Advection by shear flows
        P_adv = self._advect(P)

        # (2) Diffusion
        lap = self._laplacian(P)
        P_dif = P + self.D*lap

        # (3) Base update (explicit Euler)
        P_new = 0.5*P_adv + 0.5*P_dif

        # (4) Linear mode sources + RWM + nonlinear saturation
        amps = self.measure_modes()

        # harmonic source
        S_modes = np.zeros_like(P)
        for (m,n), g0 in self.gamma0.items():
            if enable_growth:
                S_modes += g0 * amps[(m,n)] * self._harmonic_basis(m,n)

        # RWM drive
        RW = self._rwm_drive(amps)
        for (m,n), r in RW.items():
            S_modes += r * self._harmonic_basis(m,n) * self.r_edge_mask

        # Nonlinear mode coupling
        CP = self._mode_coupling(amps)
        for (m,n), c in CP.items():
            S_modes += c * self._harmonic_basis(m,n)

        # Nonlinear saturation
        S_nl = - self.mu * (P**3)

        # (5) Coil field (control)
        if enable_control:
            coil_field = np.tensordot(self.u, self.coil_fields, axes=(0,0))
            P_new += coil_gain * coil_field

        # (6) Add sources & saturation
        P_new += self.dt * ( S_modes + S_nl )

        # (7) Noise
        if noise_on and self.noise_sigma > 0:
            P_new += np.random.normal(0.0, self.noise_sigma, size=P_new.shape)

        # (8) Disruption event (optional, impulsive)
        if disruption is not None and disruption.get('active', False):
            mag = float(disruption.get('delta', 0.5))
            reg = disruption.get('region', 'edge')
            mask = np.ones_like(P_new)
            if reg == 'edge': mask = self.r_edge_mask
            elif reg == 'core': mask = 1.0 - self.r_edge_mask
            P_new += mag * mask

        # Update state
        self.P = P_new

        # History
        amps2 = self.measure_modes()
        self.h["t"].append(len(self.h["t"]))
        self.h["A11"].append(amps2[(1,1)])
        self.h["A21"].append(amps2[(2,1)])
        self.h["A22"].append(amps2[(2,2)])
        self.h["u_norm"].append(float(np.linalg.norm(self.u)))
        self.h["rwm"].append(float(self.rwm_state))

        return amps2

# ------------------ Closed-Loop Harness -----------------------
def run_fusion_mhd_demo(steps=150,
                        urt_alpha=1.0, urt_theta_h=2.0, urt_beta=0.30,
                        urt_inner_loops=5, kappa_max=0.95,
                        R0=1.7, a=0.5, Nr=24, Nth=48, Nph=48, seed=1,
                        # physics
                        D=5e-3, mu=0.08, dt=1.0, q0=1.2, q_edge=3.8,
                        rwm_gain=0.03, tau_wall=150.0,
                        noise_sigma=2e-4, couple_strength=0.05,
                        # control
                        coil_gain=0.06, weights=(1.0, 0.8, 0.6),
                        # events
                        disruption_step=60, disruption_delta=0.8, disruption_region='edge',
                        show_progress=True):
    print("Initializing MHD-inspired tokamak + URT controller …")
    urt = FastURT(alpha=urt_alpha, theta_h=urt_theta_h, beta=urt_beta,
                  kappa_max=kappa_max, inner_loops=urt_inner_loops)
    sim = Tokamak3D_MHD(R0=R0, a=a, Nr=Nr, Nth=Nth, Nph=Nph, seed=seed,
                        D=D, mu=mu, dt=dt, q0=q0, q_edge=q_edge,
                        rwm_gain=rwm_gain, tau_wall=tau_wall,
                        noise_sigma=noise_sigma, couple_strength=couple_strength)

    # Coil mapping: harmonics (3) → coils (4)
    M = sim.coil_to_harmonics_map()         # (3,4)
    Minv = np.linalg.pinv(M)                # (4,3)
    w = np.array(weights, dtype=float).reshape(3)

    disruption = {'active': False}
    for k in range(steps):
        # Trigger a synthetic "ELM/disruption"
        if (disruption_step is not None) and (k == int(disruption_step)):
            disruption = {'active': True, 'delta': disruption_delta, 'region': disruption_region}
        else:
            disruption = {'active': False}

        amps = sim.step(enable_growth=True, enable_control=True,
                        coil_gain=coil_gain, noise_on=True,
                        disruption=disruption)
        a_vec = np.array([amps[(1,1)], amps[(2,1)], amps[(2,2)]])
        # control objective: cancel measured harmonics (weighted)
        v = Minv @ (-w * a_vec)

        # URT update for coil currents
        sim.u = urt.step(sim.u, v)

        if show_progress and ((k+1) % max(1, steps//6) == 0 or k==0):
            print(f" step {k+1:>3d} | A11={a_vec[0]: .3e}  A21={a_vec[1]: .3e}  A22={a_vec[2]: .3e} | "
                  f"‖u‖={np.linalg.norm(sim.u):.3f} | RWM={sim.rwm_state:+.3e}")

    # Final amplitudes
    ampsF = sim.measure_modes()
    print("\nFinal amplitudes:")
    for key in sim.modes:
        print(f"  A{key} = {ampsF[key]: .3e}")
    print(f"Final RWM state: {sim.rwm_state:+.3e}")
    return sim

# -------------------- Open-loop baseline ----------------------
def run_open_loop_mhd(steps=150, **sim_kwargs):
    sim = Tokamak3D_MHD(**sim_kwargs)
    for k in range(steps):
        sim.step(enable_growth=True, enable_control=False, coil_gain=0.0, noise_on=True)
    return sim

# ----------------------- Visualization -----------------------
def show_isosurface(sim, iso=0.40, title="3D Plasma Isosurface"):
    P = sim.P.copy()
    P = (P - P.min()) / (P.max() - P.min() + 1e-12)

    if HAS_PLOTLY:
        print("Rendering Plotly isosurface …")
        fig = go.Figure(data=go.Isosurface(
            x=sim.X.flatten(), y=sim.Y.flatten(), z=sim.Z.flatten(),
            value=P.flatten(),
            isomin=iso, isomax=iso,
            surface_count=1,
            caps=dict(x_show=False, y_show=False, z_show=False),
        ))
        fig.update_layout(
            title=f"{title} (iso={iso:.2f})",
            scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z",
                       aspectmode="data"),
            width=900, height=700
        )
        fig.show()
    else:
        print("Plotly not found. Falling back to Matplotlib 3D scatter.")
        mask = P > iso
        x, y, z = sim.X[mask], sim.Y[mask], sim.Z[mask]
        if x.size > 40000:
            idx = np.random.choice(x.size, 40000, replace=False)
            x, y, z = x.flatten()[idx], y.flatten()[idx], z.flatten()[idx]
        fig = plt.figure(figsize=(8,7))
        ax = fig.add_subplot(111, projection='3d')
        ax.scatter(x, y, z, s=1, alpha=0.35)
        ax.set_title(f"{title} (iso={iso:.2f})")
        ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
        ax.view_init(22, 35)
        plt.show()

def plot_metrics(sim, label="(URT on)"):
    t   = np.array(sim.h["t"], dtype=float)
    A11 = np.abs(np.array(sim.h["A11"]))
    A21 = np.abs(np.array(sim.h["A21"]))
    A22 = np.abs(np.array(sim.h["A22"]))
    un  = np.array(sim.h["u_norm"])
    rwm = np.array(sim.h["rwm"])

    fig, ax = plt.subplots(3,1, figsize=(9,8), sharex=True)
    ax[0].plot(t, A11, label="|A(1,1)|")
    ax[0].plot(t, A21, label="|A(2,1)|")
    ax[0].plot(t, A22, label="|A(2,2)|")
    ax[0].set_yscale("log")
    ax[0].set_ylabel("Mode amplitude (log)")
    ax[0].grid(True, alpha=0.3); ax[0].legend()

    ax[1].plot(t, un, label="‖u‖ (coil current)")
    ax[1].grid(True, alpha=0.3); ax[1].set_ylabel("Control norm")

    ax[2].plot(t, rwm, label="RWM state")
    ax[2].grid(True, alpha=0.3); ax[2].set_xlabel("Step"); ax[2].set_ylabel("RWM proxy")
    fig.suptitle(f"MHD-Inspired URT Metrics {label}")
    plt.tight_layout(); plt.show()

# ------------------------ Quick Summary -----------------------
def summarize(sim, name):
    amps = sim.measure_modes()
    s = f"{name}: " + ", ".join([f"A{mn}={amps[mn]:.3e}" for mn in sim.modes]) + f", RWM={sim.rwm_state:+.3e}"
    print(s)

# ============================ RUN =============================
# You can safely tweak params below. URT auto-stabilizes κ<kappa_max.
params = dict(
    steps=150,
    # URT
    urt_alpha=1.0, urt_theta_h=2.0, urt_beta=0.30, urt_inner_loops=7, kappa_max=0.95,
    # Grid / geometry
    R0=1.7, a=0.5, Nr=24, Nth=48, Nph=48, seed=3,
    # Physics
    D=5e-3, mu=0.08, dt=1.0, q0=1.2, q_edge=3.8,
    rwm_gain=0.035, tau_wall=160.0,
    noise_sigma=2e-4, couple_strength=0.06,
    # Control
    coil_gain=0.06, weights=(1.00, 0.85, 0.70),
    # Disruption event
    disruption_step=60, disruption_delta=0.9, disruption_region='edge',
    show_progress=True
)

# Closed-loop
sim_cl = run_fusion_mhd_demo(**params)
plot_metrics(sim_cl, label="(URT on)")
show_isosurface(sim_cl, iso=0.42, title="Closed-loop (URT)")

# Open-loop comparison (same physics, no control)
print("\nRunning open-loop baseline …")
sim_ol = run_open_loop_mhd(steps=params["steps"],
                           R0=params["R0"], a=params["a"],
                           Nr=params["Nr"], Nth=params["Nth"], Nph=params["Nph"], seed=params["seed"],
                           D=params["D"], mu=params["mu"], dt=params["dt"],
                           q0=params["q0"], q_edge=params["q_edge"],
                           rwm_gain=params["rwm_gain"], tau_wall=params["tau_wall"],
                           noise_sigma=params["noise_sigma"], couple_strength=params["couple_strength"])
plot_metrics(sim_ol, label="(open loop)")
show_isosurface(sim_ol, iso=0.42, title="Open-loop")

# Summaries
summarize(sim_cl, "Closed loop (URT)")
summarize(sim_ol, "Open loop     ")

Output hidden; open in https://colab.research.google.com to view.